In [ ]:
! pip install -q "datasets<4.0.0"  transformers seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 11.9 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("jakartaresearch/semeval-absa", "restaurant")


README.md:   0%|          | 0.00/3.56k [00:00<?, ?B/s]

semeval-absa.py:   0%|          | 0.00/5.49k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/231k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/69.6k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3044 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/800 [00:00<?, ? examples/s]

In [ ]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'aspects', 'category'],
        num_rows: 3044
    })
    validation: Dataset({
        features: ['id', 'text', 'aspects', 'category'],
        num_rows: 800
    })
})


In [ ]:
dataset["train"]["aspects"][1]

{'term': ['food'], 'polarity': ['positive'], 'from': [57], 'to': [61]}

In [ ]:
def format_input(batch):

    batch_labels = []

    texts = batch["text"]
    aspects = batch["aspects"]
    num_examples = len(texts)

    for i in range(num_examples):
        terms = aspects[i]["term"]
        polarities = aspects[i]["polarity"]
        pairs = []

        for term, polarity in zip(terms, polarities):
            if term != "" and polarity != "":
                pairs.append(f"{term.strip()}: {polarity.strip()}")

        if not pairs:
            target = "NONE"
        else:
            target = " | ".join(pairs)

        batch_labels.append(target)

    return {
        "input_text": texts,
        "labels": batch_labels,
    }

format_dataset = dataset.map(
    format_input,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

Map:   0%|          | 0/3044 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

In [ ]:
format_dataset

DatasetDict({
    train: Dataset({
        features: ['input_text', 'labels'],
        num_rows: 3044
    })
    validation: Dataset({
        features: ['input_text', 'labels'],
        num_rows: 800
    })
})

In [ ]:
MODEL_NAME = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

In [ ]:
def tokenized(batch):
    tokenized_inputs = tokenizer(
        batch["input_text"],
        max_length=128,
        truncation=True,

    )

    tokenized_labels = tokenizer(
        batch["labels"],
        max_length=64,
        truncation=True
    )

    tokenized_inputs["labels"] = tokenized_labels["input_ids"]

    return tokenized_inputs

tokenized_datasets = format_dataset.map(
    tokenized,
    batched=True,
    remove_columns=format_dataset["train"].column_names
)

Map:   0%|          | 0/3044 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained("NUSTM/restaurant-t5-base")

config.json:   0%|          | 0.00/1.48k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100
)

training_args = Seq2SeqTrainingArguments(
    output_dir='/content/t5_restaurant_absa',

    # 1. Tham số đặc thù quan trọng của Seq2Seq / T5
    predict_with_generate=True,
    generation_max_length=64,
    generation_num_beams=2,
    fp16=True,

    num_train_epochs=10,
    learning_rate=3e-4,
    weight_decay=0.01,
    max_grad_norm=1.0,
    warmup_steps=50,

    # Batch size (Lưu ý về VRAM bên dưới)
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,

    # Logging & Checkpointing
    logging_strategy='epoch',
    logging_first_step=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,

    report_to='none',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    predict_with_generate=False
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.973955,0.257259


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 